In [59]:
import random
import itertools
import math



def fill_rooms(size, n_rooms):
    #grid = [[]]
    #for row in range(size):
    #    grid.append([])
    #    for col in range(size):
    #        grid[row].append(random.randint(1,n_rooms))

    #disconnected_rooms = check_disconnected_rooms(grid)

    #while len(disconnected_rooms)>0:
    #    grid = [[]]
    #    for row in range(size):
    #        grid.append([])
    #        for col in range(size):
    #            grid[row].append(random.randint(1,n_rooms))
    #
    #    disconnected_rooms = check_disconnected_rooms(grid)
    #    #check 
    

    grid = []
    for row in range(size):
        grid.append([])
        for col in range(size):
            grid[row].append(0)
    for room in range(1, n_rooms+1):
        cell = select_random_cell_selecting_rooms(size, grid, cannot_have_room_selected = True)
        grid[cell[0]][cell[1]] = room
        direction = random.randint(1,4)
        if direction == 1 and cell[0] > 0: 
            if grid[cell[0]-1][cell[1]] == 0:
                grid[cell[0]-1][cell[1]] = room
            else: 
                direction = 2
        if direction == 2 and cell[0] < size - 1: 
            if grid[cell[0]+1][cell[1]] == 0:
                grid[cell[0]+1][cell[1]] = room
            else: 
                direction = 3
        if direction == 3 and cell[1] > 0: 
            if grid[cell[0]][cell[1]-1] == 0:
                grid[cell[0]][cell[1]-1] = room
            else: 
                direction = 4
        if direction == 4 and cell[1] < size - 1: 
            if grid[cell[0]][cell[1]+1] == 0:
                grid[cell[0]][cell[1]+1] = room
            else: 
                direction = 1
        if direction == 1 and cell[0] > 0: 
            if grid[cell[0]-1][cell[1]] == 0:
                grid[cell[0]-1][cell[1]] = room

    while 0 in list(set(itertools.chain(*grid))):
        cell = select_random_cell_selecting_rooms(size, grid, cannot_have_room_selected = True)
        neighbor_rooms = []
        if cell[0] > 0: 
            if grid[cell[0]-1][cell[1]] != 0:
                neighbor_rooms.append(grid[cell[0]-1][cell[1]])
        if cell[0] < size - 1: 
            if grid[cell[0]+1][cell[1]] != 0:
                neighbor_rooms.append(grid[cell[0]+1][cell[1]])
        if cell[1] > 0: 
            if grid[cell[0]][cell[1]-1] != 0:
                neighbor_rooms.append(grid[cell[0]][cell[1]-1])
        if cell[1] < size - 1: 
            if grid[cell[0]][cell[1]+1] != 0:
                neighbor_rooms.append(grid[cell[0]][cell[1]+1])
        if len(neighbor_rooms) > 0:
            grid[cell[0]][cell[1]] = random.choice(neighbor_rooms)
    return grid

def select_random_cell_selecting_rooms(size, grid, cannot_have_room_selected = False):
    if not cannot_have_room_selected:
        return [random.randint(0,size-1), random.randint(0,size-1)]
    else:
        while 0 in list(set(itertools.chain(*grid))):
            cell = [random.randint(0,size-1), random.randint(0,size-1)]
            if grid[cell[0]][cell[1]] == 0:
                return cell
        return [-1, -1]

def check_disconnected_rooms(grid):
    disconnected_list = []
    room_nums = list(set(itertools.chain(*rooms)))
    for room in room_nums:
        room_locations = find_room_locations(grid, room)
        to_check = room_locations
        blobs = [[x] for x in room_locations]
        while len(to_check) > 0:
            checking = to_check[0]
            to_check.remove(checking)
            checking_blob = []
            for blob in blobs:
                if checking in blob:
                    checking_blob = blob
            blobs.remove(checking_blob)
            if [checking[0]+1, checking[1]] in room_locations:
                joining_blob = []
                for blob in blobs:
                    if [checking[0]+1, checking[1]] in blob:
                        joining_blob = blob
                        blobs.remove(joining_blob)
                    checking_blob.extend(x for x in joining_blob if x not in checking_blob)
            if [checking[0], checking[1]+1] in room_locations:
                joining_blob = []
                for blob in blobs:
                    if [checking[0], checking[1]+1] in blob:
                        joining_blob = blob
                        blobs.remove(joining_blob)
                    checking_blob.extend(x for x in joining_blob if x not in checking_blob)
            blobs.append(checking_blob)
        if len(blobs)>1:
            disconnected_list.append(room)
    return disconnected_list


def find_room_locations(grid, room_num):
    room_locations = []
    for row in range(len(grid)):
        for col in range(len(grid[row])):
            if grid[row][col] == room_num:
                room_locations.append([row, col])
    return room_locations


def count_people_in_room(rooms, room_num, person_positions):
    n_people = 0
    for [i, j] in person_positions:
        if rooms[i][j] == room_num:
            n_people += 1
    return n_people


def place_persons(size, rooms):
    not_finished = True
    room_with_victim = 0
    room_nums = list(set(itertools.chain(*rooms)))
    num_tries = 0
    while not_finished and len(room_nums) > 1 and num_tries < 1000:
        person_positions = []
        for row in range(0, size):
            double_col = True
            while double_col:
                double_col = False
                col = random.randint(0, size-1)
                for [i, j] in person_positions:
                    if j == col: 
                        double_col = True
            person_positions.append([row, col])
        for room_num in room_nums:
            if count_people_in_room(rooms, room_num, person_positions) == 2:
                not_finished = False
                room_with_victim = room_num
        num_tries += 1
    if num_tries == 1000:
        print("Impossible room, try again.")
        exit
    not_found_victim = True
    random.shuffle(person_positions)
    for [i, j] in person_positions:
        if room_with_victim == rooms[i][j] and not_found_victim:
            not_found_victim = False
            person_positions.remove([i, j])
            person_positions.append([i, j])
    return person_positions


def place_objects(rooms, objects_can_be_occupied, objects_cannot_be_occupied, person_positions, object_rate):
    object_grid = []
    for row in range(size):
        object_grid.append([])
        for col in range(size):
            object_grid[row].append(0)

    for i in range(math.floor(object_rate * len(rooms) * len(rooms))):
        object_type = random.randint(-objects_cannot_be_occupied, objects_can_be_occupied-1)  #objects not occupied get negative index, objects that can be occupied positive
        can_be_occupied = False
        if object_type >= 0:
            can_be_occupied = True
            object_type += 1
        size_hor = 1 
        size_ver = 1
        if object_type == 1 or object_type == -1:
            if random.random() < 0.5:
                size_hor = 2
            else:
                size_ver = 2
            i += 1
        if object_type == 2:
            size_hor = random.randint(1,3)
            size_ver = random.randint(1,3)
            i += size_hor * size_ver - 1

        object_position = select_random_cell_for_object(len(rooms), rooms, person_positions, object_grid, can_be_occupied, size_hor, size_ver)
        if object_position != [-1, -1]:
            for i in range(size_hor):
                for j in range(size_ver):
                    object_grid[object_position[0]+i][object_position[1]+j] = object_type
        
    return object_grid


def select_random_cell_for_object(size, rooms, person_positions, object_grid, can_be_occupied, size_hor = 1, size_ver = 1):

    num_tries = 1
    if can_be_occupied:
        occupied_or_different_rooms = False
        cell = [random.randint(0,size-size_hor), random.randint(0,size-size_ver)]
        room = rooms[cell[0]][cell[1]]
        for i in range(size_hor):
            for j in range(size_ver):
                occupied_or_different_rooms = occupied_or_different_rooms or rooms[cell[0]+i][cell[1]+j] != room or object_grid[cell[0]+i][cell[1]+j] != 0
        while occupied_or_different_rooms:
            occupied_or_different_rooms = False
            cell = [random.randint(0,size-size_hor), random.randint(0,size-size_ver)]
            room = rooms[cell[0]][cell[1]]
            for i in range(size_hor):
                for j in range(size_ver):
                    occupied_or_different_rooms = occupied_or_different_rooms or rooms[cell[0]+i][cell[1]+j] != room or object_grid[cell[0]+i][cell[1]+j] != 0
            num_tries += 1
            if num_tries > 1000:
                return [-1, -1]
        return cell
    
    else:
        occupied_or_different_rooms = False
        cell = [random.randint(0,size-size_hor), random.randint(0,size-size_ver)]
        room = rooms[cell[0]][cell[1]]
        for i in range(size_hor):
            for j in range(size_ver):
                occupied_or_different_rooms = occupied_or_different_rooms or rooms[cell[0]+i][cell[1]+j] != room or object_grid[cell[0]+i][cell[1]+j] != 0 or [cell[0]+i, cell[1]+j] in person_positions
        while occupied_or_different_rooms:
            occupied_or_different_rooms = False
            cell = [random.randint(0,size-size_hor), random.randint(0,size-size_ver)]
            room = rooms[cell[0]][cell[1]]
            for i in range(size_hor):
                for j in range(size_ver):
                    occupied_or_different_rooms = occupied_or_different_rooms or rooms[cell[0]+i][cell[1]+j] != room or object_grid[cell[0]+i][cell[1]+j] != 0 or [cell[0]+i, cell[1]+j] in person_positions
            num_tries += 1
            if num_tries > 1000:
                return [-1, -1]
        return cell

def gender_persons(size):
    genders = ["female", "male"]
    return random.choices(genders, k=size)

def generate_clues(size, rooms, person_positions, object_positions, person_genders):
    clues = []
    for pers in range(size - 1):
        clue = generate_clue(size, rooms, person_positions, object_positions, person_genders, pers)
        clues.append(clue)
    number_solutions = check_number_solutions(size, rooms, object_positions, person_genders, clues)
    while number_solutions > 1:
        pers = random.randint(0, size - 2)
        clue = generate_clue(size, rooms, person_positions, object_positions, person_genders, pers)
        new_clues = clues.copy()
        new_clues.append(clue)
        new_number_solutions = check_number_solutions(size, rooms, object_positions, person_genders, new_clues)
        if new_number_solutions < number_solutions:
            clues = new_clues
            number_solutions = new_number_solutions
    #filter dubbele info (niet in room, wel in andere room)
    return clues

def generate_clue(size, rooms, person_positions, object_positions, person_genders, pers):
    not_found = True
    cell = person_positions[pers]
    room = rooms[cell[0]][cell[1]]

    while not_found:
        types = ['in_same_room', 'not_in_same_room', 'next_to', 'not_next_to', 'on', 'not_on', 'in_room', 'not_in_room', \
                'in_same_room', 'not_in_same_room', 'next_to', 'not_next_to', 'on', 'not_on', 'in_room', 'not_in_room', \
                'in_same_room', 'not_in_same_room', 'next_to', 'not_next_to', 'on', 'not_on', 'in_room', 'not_in_room', \
                'in_same_room', 'not_in_same_room', 'next_to', 'not_next_to', 'on', 'not_on', 'in_room', 'not_in_room', \
                'in_top_row', 'in_bottom_row', 'in_left_column', 'in_right_column', \
                'in_top_row', 'in_bottom_row', 'in_left_column', 'in_right_column', \
                'in_top_row', 'in_bottom_row', 'in_left_column', 'in_right_column', \
                'in_room_with_female', 'in_room_with_male', \
                'in_room_with_female', 'in_room_with_male', \
                'in_column_n', 'in_row_n', 'east_of', 'west_of', 'north_of', 'south_of']
        type = random.choice(types)
        clue = {'type': type, 'pers': pers}

        if type == 'in_same_room':
            pers2list = []
            for pers2 in range(size - 1):
                if pers2 != pers:
                    cell2 = person_positions[pers2]
                    room2 = rooms[cell2[0]][cell2[1]]
                    if room == room2:
                        pers2list.append(pers2)
            if len(pers2list) > 0:
                clue["pers2"] = random.choice(pers2list)
                return clue
        if type == 'not_in_same_room':
            pers2list = []
            for pers2 in range(size - 1):
                if pers2 != pers:
                    cell2 = person_positions[pers2]
                    room2 = rooms[cell2[0]][cell2[1]]
                    if room != room2:
                        pers2list.append(pers2)
            clue["pers2"] = random.choice(pers2list)
            return clue
        if type == 'east_of':
            pers2list = []
            for pers2 in range(size - 1):
                cell2 = person_positions[pers2]
                if pers2 != pers and cell[1] > cell2[1]:
                    pers2list.append(pers2)
            clue["pers2"] = random.choice(pers2list)
            return clue
        if type == 'west_of':
            pers2list = []
            for pers2 in range(size - 1):
                cell2 = person_positions[pers2]
                if pers2 != pers and cell[1] < cell2[1]:
                    pers2list.append(pers2)
            clue["pers2"] = random.choice(pers2list)
            return clue
        if type == 'north_of':
            pers2list = []
            for pers2 in range(size - 1):
                cell2 = person_positions[pers2]
                if pers2 != pers and cell[0] < cell2[0]:
                    pers2list.append(pers2)
            clue["pers2"] = random.choice(pers2list)
            return clue
        if type == 'south_of':
            pers2list = []
            for pers2 in range(size - 1):
                cell2 = person_positions[pers2]
                if pers2 != pers and cell[0] > cell2[0]:
                    pers2list.append(pers2)
            clue["pers2"] = random.choice(pers2list)
            return clue
        if type == 'next_to':
            objectlist = []
            if cell[0] < size - 1 and object_positions[cell[0]+1][cell[1]] != 0:
                objectlist.append(object_positions[cell[0]+1][cell[1]])
            if cell[0] > 0 and object_positions[cell[0]-1][cell[1]] != 0:
                objectlist.append(object_positions[cell[0]-1][cell[1]])
            if cell[1] < size - 1 and object_positions[cell[0]][cell[1]+1] != 0:
                objectlist.append(object_positions[cell[0]][cell[1]+1])
            if cell[1] > 0 and object_positions[cell[0]][cell[1]-1] != 0:
                objectlist.append(object_positions[cell[0]][cell[1]-1])
            clue["obj"] = random.choice(objectlist)
            return clue
        if type == 'not_next_to':            
            objectlist = [0]
            if cell[0] < size - 1 and object_positions[cell[0]+1][cell[1]] != 0:
                objectlist.append(object_positions[cell[0]+1][cell[1]])
            if cell[0] > 0 and object_positions[cell[0]-1][cell[1]] != 0:
                objectlist.append(object_positions[cell[0]-1][cell[1]])
            if cell[1] < size - 1 and object_positions[cell[0]][cell[1]+1] != 0:
                objectlist.append(object_positions[cell[0]][cell[1]+1])
            if cell[1] > 0 and object_positions[cell[0]][cell[1]-1] != 0:
                objectlist.append(object_positions[cell[0]][cell[1]-1])
            notobjectlist = [x for x in list(set(itertools.chain(*object_positions))) if x not in objectlist]
            clue["obj"] = random.choice(notobjectlist)
            return clue
        if type == 'on':
            if object_positions[cell[0]][cell[1]] != 0:
                clue['obj'] = object_positions[cell[0]][cell[1]]
                return clue
        if type == 'not_on':
            if object_positions[cell[0]][cell[1]] != 0:
                objects = [x for x in list(set(itertools.chain(*object_positions))) if x > 0 and x != object_positions[cell[0]][cell[1]]]
                clue['obj'] = random.choice(objects)
                return clue
        if type == 'in_room':
            clue['room'] = rooms[cell[0]][cell[1]]
            return clue
        if type == 'not_in_room':
            room_nums = list(set(itertools.chain(*rooms)))
            room_nums.remove(rooms[cell[0]][cell[1]])
            clue['room'] = random.choice(room_nums) 
            return clue
        if type == 'in_room_with_female':
            if in_room_with_female(size, rooms, person_positions, person_genders, pers):
                return clue
        if type == 'in_room_with_male':
            if in_room_with_male(size, rooms, person_positions, person_genders, pers):
                return clue
        if type == 'in_left_column':
            if cell[1] == 0:
                return clue
        if type == 'in_right_column':
            if cell[1] == size - 1:
                return clue
        if type == 'in_top_row':
            if cell[0] == 0:
                return clue
        if type == 'in_bottom_row':
            if cell[0] == size - 1:
                return clue
        if type == 'in_column_n':
            clue["n"] = cell[1]
            return clue
        if type == 'in_row_n':
            clue["n"] = cell[0]
            return clue
    return clue

def check_number_solutions(size, rooms, object_positions, person_genders, clues):
    n_solutions = 0
    rows = list(itertools.permutations(range(size)))
    cols = list(itertools.permutations(range(size)))
    for row in rows:
        for col in cols:
            person_positions_test = []
            for i in range(size):
                person_positions_test.append([row[i], col[i]])
            pos_victim = person_positions_test[size-1]
            room_victim = rooms[pos_victim[0]][pos_victim[1]]
            people_in_room_victim = count_people_in_room(rooms, room_victim, person_positions_test)
            is_solution = people_in_room_victim != 2
            if len(clues) > 0:
                for clue in clues:
                    is_solution = is_solution and not check_clue(size, rooms, person_positions_test, object_positions, person_genders, clue)
            if is_solution: 
                n_solutions += 1
    return n_solutions






In [55]:
listje = []
list2 = listje.copy()
list2.append('hallo')

print(listje)
print(list2)

[]
['hallo']


In [56]:
def check_clue(size, rooms, person_positions, object_positions, person_genders, clue):
    type = clue["type"]
    if type == 'in_same_room':
        return in_same_room(rooms, person_positions, clue["pers"], clue["pers2"])
    if type == 'not_in_same_room':
        return not_in_same_room(rooms, person_positions, clue["pers"], clue["pers2"])
    if type == 'next_to':
        return next_to(size, rooms, person_positions, object_positions, clue["pers"], clue["obj"])
    if type == 'not_next_to':
        return not_next_to(size, rooms, person_positions, object_positions, clue["pers"], clue["obj"])
    if type == 'on':
        return on(person_positions, object_positions, clue["pers"], clue["obj"])
    if type == 'not_on':
        return not_on(person_positions, object_positions, clue["pers"], clue["obj"])
    if type == 'in_room':
        return in_room(rooms, person_positions, clue["pers"], clue["room"])
    if type == 'not_in_room':
        return in_room(rooms, person_positions, clue["pers"], clue["room"])
    if type == 'in_room_with_female':
        return in_room_with_female(size, rooms, person_positions, person_genders, clue["pers"])
    if type == 'in_room_with_male':
        return in_room_with_male(size, rooms, person_positions, person_genders, clue["pers"])
    if type == 'in_left_column':
        return in_left_column(size, person_positions, clue["pers"])
    if type == 'in_right_column':
        return in_right_column(size, person_positions, clue["pers"])
    if type == 'in_top_row':
        return in_top_row(size, person_positions, clue["pers"])
    if type == 'in_bottom_row':
        return in_bottom_row(size, person_positions, clue["pers"])
    if type == 'in_column_n':
        return in_column_n(person_positions, clue["pers"], clue["n"])
    if type == 'in_row_n':
        return in_row_n(person_positions, clue["pers"], clue["n"])
    if type == 'east_of':
        return east_of(person_positions, clue["pers"], clue["pers2"])
    if type == 'west_of':
        return west_of(person_positions, clue["pers"], clue["pers2"])
    if type == 'north_of':
        return north_of(person_positions, clue["pers"], clue["pers2"])
    if type == 'south_of':
        return south_of(person_positions, clue["pers"], clue["pers2"])
    return False




def in_same_room(rooms, person_positions, pers, pers2):
    cellA = person_positions[pers]
    cellB = person_positions[pers2]
    return rooms[cellA[0]][cellA[1]] == rooms[cellB[0]][cellB[1]]

def not_in_same_room(rooms, person_positions, pers, pers2):
    return not in_same_room(rooms, person_positions, pers, pers2)

def next_to(size, rooms, person_positions, object_positions, pers, obj):
    cell = person_positions[pers]
    next_to = False
    if cell[0] < size - 1:
        next_to = next_to or rooms[cell[0]][cell[1]] == rooms[cell[0]+1][cell[1]] and object_positions[cell[0]+1][cell[1]] == obj 
    if cell[0] > 0:
        next_to = next_to or rooms[cell[0]][cell[1]] == rooms[cell[0]-1][cell[1]] and object_positions[cell[0]-1][cell[1]] == obj
    if cell[1] < size - 1:
        next_to = next_to or rooms[cell[0]][cell[1]+1] and object_positions[cell[0]][cell[1]+1] == obj
    if cell[1] > 0:
        next_to = next_to or rooms[cell[0]][cell[1]-1] and object_positions[cell[0]][cell[1]-1] == obj
    return next_to

def not_next_to(size, rooms, person_positions, object_positions, pers, obj):
    return not next_to(size, rooms, person_positions, object_positions, pers, obj)

def on(person_positions, object_positions, pers, obj):
    cell = person_positions[pers]
    return object_positions[cell[0]][cell[1]] == obj

def not_on(person_positions, object_positions, pers, obj):
    return not on(person_positions, object_positions, pers, obj)

def in_corner(size, rooms, person_positions, pers):
    cell = person_positions[pers]
    against_vertical_wall = False
    if cell[1] == 0 or cell[1] == size - 1:
        against_vertical_wall = True
    else:
        if rooms[cell[0]][cell[1]] != rooms[cell[0]][cell[1]-1] or rooms[cell[0]][cell[1]] != rooms[cell[0]][cell[1]+1]:
            against_vertical_wall = True
    against_horizontal_wall = False
    if cell[0] == 0 or cell[0] == size - 1:
        against_horizontal_wall = True
    else:
        if rooms[cell[0]][cell[1]] != rooms[cell[0]-1][cell[1]] or rooms[cell[0]][cell[1]] != rooms[cell[0]+1][cell[1]]:
            against_horizontal_wall = True
    return against_horizontal_wall and against_vertical_wall

def not_in_corner(size, rooms, person_positions, pers):
    return not in_corner(size, rooms, person_positions, pers)

def in_room(rooms, person_positions, pers, room):
    cell = person_positions[pers]
    return rooms[cell[0]][cell[1]] == room

def not_in_room(rooms, person_positions, pers, room):
    return not in_room(rooms, person_positions, pers, room)

def in_room_with_female(size, rooms, person_positions, person_genders, pers):
    cellA = person_positions[pers]
    for i in range(size):
        cellB = person_positions[i]
        if i != pers and person_genders[i] == 'female' and rooms[cellA[0]][cellA[1]] == rooms[cellB[0]][cellB[1]]:
            return True
    return False

def in_room_with_male(size, rooms, person_positions, person_genders, pers):
    cellA = person_positions[pers]
    for i in range(size):
        cellB = person_positions[i]
        if i != pers and person_genders[i] == 'male' and rooms[cellA[0]][cellA[1]] == rooms[cellB[0]][cellB[1]]:
            return True
    return False

def in_left_column(size, person_positions, pers):
    return in_column_n(person_positions, pers, 0)

def in_right_column(size, person_positions, pers):
    return in_column_n(person_positions, pers, size - 1)

def in_top_row(size, person_positions, pers):
    return in_row_n(person_positions, pers, 0)

def in_bottom_row(size, person_positions, pers):
    return in_row_n(person_positions, pers, size - 1)

def in_column_n(person_positions, pers, n):
    cell = person_positions[pers]
    return cell[1] == n

def in_row_n(person_positions, pers, n):
    cell = person_positions[pers]
    return cell[0] == n

def east_of(person_positions, pers, pers2):
    cellA = person_positions[pers]
    cellB = person_positions[pers2]
    return cellA[1] > cellB[1] 

def west_of(person_positions, pers, pers2):
    return not east_of(person_positions, pers, pers2)

def north_of(person_positions, pers, pers2):
    cellA = person_positions[pers]
    cellB = person_positions[pers2]
    return cellA[0] < cellB[0] 

def south_of(person_positions, pers, pers2):
    return not north_of(person_positions, pers, pers2)



In [64]:
random.seed(0)

size = 6

n_rooms = random.randint(size - 3, size)
objects_can_be_occupied = 4 #The first is always size 2x1, the second will be randint(1,3) x randint(1,3)
objects_cannot_be_occupied = 3 #The first is always size 2x1
object_rate = 0.33

rooms = fill_rooms(size, n_rooms) 

print(f"Number of rooms: {n_rooms}")
for row in range(size):
    print(rooms[row])

print('\nLocation persons:')
person_positions = place_persons(size, rooms)
print(person_positions)

print('\nGenders:')
person_genders = gender_persons(size)
print(person_genders)

print('\nMap of objects:')
object_positions = place_objects(rooms, objects_can_be_occupied, objects_cannot_be_occupied, person_positions, object_rate)
for row in range(size):
    print(object_positions[row])

print('\nClues:')
clues = generate_clues(size, rooms, person_positions, object_positions, person_genders)
for clue in clues:
    print(clue)

nsol = check_number_solutions(size, rooms, object_positions, person_genders, clues)
print(f'\nSolutions: {nsol}')

Number of rooms: 6
[5, 5, 3, 3, 3, 2]
[5, 5, 3, 3, 2, 2]
[1, 5, 3, 3, 2, 2]
[1, 1, 3, 2, 2, 2]
[1, 4, 6, 2, 2, 2]
[1, 4, 6, 2, 2, 2]

Location persons:
[[5, 3], [2, 5], [4, 2], [0, 1], [3, 4], [1, 0]]

Genders:
['male', 'female', 'female', 'male', 'male', 'male']

Map of objects:
[4, 0, 0, -1, -1, 0]
[3, 3, -3, 0, 0, -3]
[0, 0, -1, -1, 0, 1]
[-3, 0, 0, 0, 0, 1]
[0, 0, 0, -3, 0, -1]
[0, 0, 0, 0, 0, -1]

Clues:
{'type': 'in_same_room', 'pers': 0, 'pers2': 1}
{'type': 'on', 'pers': 1, 'obj': 1}
{'type': 'west_of', 'pers': 2, 'pers2': 0}
{'type': 'in_room_with_male', 'pers': 3}
{'type': 'in_room', 'pers': 4, 'room': 2}
{'type': 'not_in_same_room', 'pers': 0, 'pers2': 3}

Solutions: 0
